In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [ ]:

df = pd.read_csv("Dataset/fake reviews dataset.csv")

print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.columns)


In [ ]:

# Keep required columns
df = df[['text_', 'label']].copy()

# Remove missing values
df = df.dropna()

# Remove duplicate reviews
df = df.drop_duplicates(subset='text_')

# Convert labels into numbers
df['label'] = df['label'].map({
    'CG': 1,
    'OR': 0
})

df = df.dropna()

# Clean text
df['text_'] = df['text_'].str.lower()
df['text_'] = df['text_'].str.replace(
    r'[^a-z\s]', ' ', regex=True
)

df['text_'] = df['text_'].str.replace(
    r'\s+', ' ', regex=True
).str.strip()

df = df[df['text_'] != '']

print(df.head())


In [ ]:

print(df['label'].value_counts())

sns.countplot(x='label', data=df)
plt.title("Fake vs Genuine Reviews")
plt.xlabel("0 = Genuine, 1 = Fake")
plt.ylabel("Number of Reviews")
plt.show()


In [ ]:

df['review_length'] = df['text_'].str.len()

sns.histplot(df['review_length'], bins=30)
plt.title("Review Length Distribution")
plt.xlabel("Number of Characters")
plt.show()


In [ ]:

X = df['text_']
y = df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing:", len(X_test))


In [ ]:

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(X_train_vec.shape)


In [ ]:

rng = np.random.RandomState(42)

# Initially mark everything as unlabeled
pu_labels = np.zeros(len(y_train), dtype=int)

# Find fake review positions
fake_indices = np.where(y_train.to_numpy() == 1)[0]

# Reveal 20% of fake reviews
known_fake = rng.choice(
    fake_indices,
    size=max(1, int(len(fake_indices) * 0.2)),
    replace=False
)

pu_labels[known_fake] = 1

print("Known Fake:", sum(pu_labels == 1))
print("Unlabeled:", sum(pu_labels == 0))


In [ ]:

# First model: Known Fake vs Unlabeled
initial_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

initial_model.fit(X_train_vec, pu_labels)

# Probability of being known positive
scores = initial_model.predict_proba(
    X_train_vec
)[:, 1]

# Select low-scoring unlabeled reviews
unlabeled_indices = np.where(pu_labels == 0)[0]

n_negative = min(
    len(known_fake),
    len(unlabeled_indices)
)

reliable_negative = unlabeled_indices[
    np.argsort(scores[unlabeled_indices])[:n_negative]
]

# Combine known positives and reliable negatives
selected = np.concatenate([
    known_fake,
    reliable_negative
])

final_labels = np.concatenate([
    np.ones(len(known_fake)),
    np.zeros(len(reliable_negative))
])

# Train final classifier
model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec[selected], final_labels)

print("PU Model Trained Successfully!")


In [ ]:

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test, y_pred,
    target_names=['Genuine', 'Fake']
))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Genuine', 'Fake'],
    yticklabels=['Genuine', 'Fake']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:

# Feature Importance
feature_names = tfidf.get_feature_names_out()

importance = model.coef_[0]

feature_df = pd.DataFrame({
    'Word': feature_names,
    'Importance': importance
})

# Top 10 words associated with fake reviews
top_features = feature_df.nlargest(
    10, 'Importance'
)

print(top_features)

# Plot feature importance
plt.figure(figsize=(10, 5))

sns.barplot(
    data=top_features,
    x='Importance',
    y='Word'
)

plt.title("Top 10 Important Words for Fake Reviews")
plt.show()


In [ ]:

# Step 12: Test Custom Review

import re

review = "This product is amazing and works perfectly!"

# Clean the review
cleaned = review.lower()
cleaned = re.sub(r'[^a-z\s]', ' ', cleaned)
cleaned = re.sub(r'\s+', ' ', cleaned).strip()

# Convert review into numerical features
review_vector = tfidf.transform([cleaned])

# Predict
prediction = model.predict(review_vector)[0]

if prediction == 1:
    print("Potentially Fake Review")
else:
    print("Likely Genuine Review")


In [ ]:

# Step 13: Save Model

import joblib

joblib.dump(model, "fake_review_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and Vectorizer Saved Successfully!")


In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
%pip install streamlit

In [ ]:


%%writefile app.py

import streamlit as st
import joblib
import re

# PAGE SETTINGS
st.set_page_config(
    page_title="ReviewGuard AI",
    page_icon="🔍",
    layout="centered"
)

# LOAD MODEL
model = joblib.load("fake_review_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

# CUSTOM DESIGN
st.markdown("""
<style>
.stApp {
    background-color: #0C1424;
    color: white;
}

.block-container {
    padding-top: 3rem;
    max-width: 850px;
}

h1, h2, h3 {
    color: #FFFFFF !important;
}

.stButton > button {
    background-color: #3B82F6;
    color: white;
    border: none;
    border-radius: 10px;
    padding: 12px 25px;
    font-weight: bold;
    width: 100%;
}

.stButton > button:hover {
    background-color: #2563EB;
    color: white;
}

.stTextArea textarea {
    border-radius: 12px;
}
</style>
""", unsafe_allow_html=True)

# HEADER
st.title("🔍 ReviewGuard AI")

st.write(
    "### Detect Suspicious Reviews with AI"
)

st.caption(
    "Powered by Semi-Supervised PU Learning, "
    "TF-IDF and Logistic Regression"
)

st.divider()

# REVIEW INPUT
st.subheader("Analyze a Product Review")

review = st.text_area(
    "Enter your review:",
    placeholder="Paste a product review here...",
    height=160
)

# PREDICTION
if st.button("🔍 Analyze Review"):

    if review.strip():

        # Clean input
        cleaned = review.lower()
        cleaned = re.sub(
            r'[^a-z\s]', ' ', cleaned
        )
        cleaned = re.sub(
            r'\s+', ' ', cleaned
        ).strip()

        if cleaned:

            # TF-IDF transformation
            review_vector = tfidf.transform(
                [cleaned]
            )

            # Model prediction
            prediction = model.predict(
                review_vector
            )[0]

            # Display result
            st.divider()
            st.subheader("Analysis Result")

            if prediction == 1:
                st.error(
                    "⚠️ Potentially Fake Review"
                )
            else:
                st.success(
                    "✅ Likely Genuine Review"
                )

            st.info(
                "This is an AI prediction, "
                "not a verified fact."
            )

        else:
            st.warning(
                "Please enter valid review text."
            )

    else:
        st.warning(
            "Please enter a review first."
        )

# ABOUT SECTION
st.divider()

st.subheader("About the Model")

col1, col2, col3 = st.columns(3)

with col1:
    st.info("🧠 PU Learning")

with col2:
    st.info("📝 TF-IDF")

with col3:
    st.info("📊 Logistic Regression")

st.caption(
    "University Machine Learning Project | "
    "Fake Review Detection System"
)


In [ ]:
import os

print(os.listdir())

In [ ]:

import subprocess
import sys

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py"
])

print("Streamlit started!")


In [ ]:
print(process.poll())

In [ ]:
!python -m streamlit run app.py

In [ ]:

import pandas as pd

# Check predictions on test dataset
predictions = model.predict(X_test_vec)

print("Prediction Counts:")
print(pd.Series(predictions).value_counts())

print("\nActual Labels:")
print(y_test.value_counts())


In [ ]:

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test,
    predictions,
    target_names=["Genuine", "Fake"]
))

print(confusion_matrix(y_test, predictions))


In [ ]:
# Select actual genuine reviews
genuine_reviews = df[df['label'] == 0]

# Show 5 random genuine reviews
print(genuine_reviews['text_'].sample(10, random_state=42).to_string(index=False))